# Baseline Evaluation on In-The-Wild — Augmentation Robustness
Evaluates pretrained deepfake detectors on `mueller91/In-The-Wild`.

| Model file | Architecture |
|---|---|
| `aasist.pth` | AASIST — sinc-filter + graph attention |
| `xlsr2.pt` | XLS-R (wav2vec2-xls-r-300m) + binary head |
| `whisper.pth` | Whisper encoder + binary head |
| `robust_goat.pth` / `robust_goat.ckpt` | WavLM + Phoneme-GAT (20k balanced, multi-aug robust) |

**Data:** 5 000 ITW samples, 50 % real / 50 % fake — evaluation only (no training).  
**Augmentations:** same full sweep as `validation.ipynb`.  
**Output:** `itw_eval_results.csv` + summary tables.

**Prerequisites:**
- `aasist.pth`, `xlsr2.pt`, `whisper.pth`, `robust_goat.pth` (or `.ckpt`) in same directory ✅
- `phoneme_GAT/modules.py` on `sys.path` (adjust `PHONEME_GAT_ROOT` below if needed) ✅
- AASIST architecture cloned in Cell 3 (code only, no weights needed from git)

> **FIX LOG**
> - `xlsr_forward` was commented out via triple-quoted string but `xlsr_forward` was never defined → `NameError` if accidentally un-commented. Replaced with proper `# commented` block with stub guard.
> - `num_workers=4` in val_dataloader causes pickling errors on many systems with `InMemoryDataset`; lowered to `num_workers=0`.
> - `pin_memory=True` with `num_workers=0` is a no-op but harmless; kept for aug_loader.
> - `evaluate_model` squeezed batch audio on `.squeeze(1)` but `InMemoryDataset` returns `[1, T]` shape — squeeze is correct; noted explicitly.
> - Summary cell (14) re-imports pandas (already imported in cell 2) — harmless but cleaned up.
> - `robust_goat` section added (cells 12a–12b): loads `Phoneme_GAT_lit` from `robust_goat.pth` or `.ckpt`, wraps forward pass to match `[B,T]→[B,2]` interface, registers in `MODEL_REGISTRY`.


In [1]:
# ── 0. Installs (run once) ───────────────────────────────────────────────────
import subprocess, sys
for pkg in ["openai-whisper", "transformers", "remotezip"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)
print("✅ dependencies ready")



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/bin/python -m pip install --upgrade pip


✅ dependencies ready



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/bin/python -m pip install --upgrade pip


In [2]:
# ── 1. Imports & torch.load monkeypatch ──────────────────────────────────────
import os, sys, json, csv, io, subprocess
from argparse import Namespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import librosa
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, roc_curve
from datasets import load_dataset, Audio
from remotezip import RemoteZip

# ── monkeypatch torch.load (Torch 2.6+) ──────────────────────────────────────
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([Namespace, Phonemer_Tokenizer_Recombination, Series])

_orig_torch_load = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_compat

torch.set_float32_matmul_precision("medium")
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16_000
CLIP_LEN  = 64_600   # 4.025 s — AASIST / RawGAT standard

# ── Path to your phoneme_GAT package ─────────────────────────────────────────
# Adjust this to wherever phoneme_GAT/modules.py lives relative to this notebook.
PHONEME_GAT_ROOT = "."   # e.g. "/path/to/project" or "."
if os.path.abspath(PHONEME_GAT_ROOT) not in sys.path:
    sys.path.insert(0, os.path.abspath(PHONEME_GAT_ROOT))

print(f"✅ Imports OK  |  device={DEVICE}  |  clip_len={CLIP_LEN}")


/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✅ Imports OK  |  device=cuda  |  clip_len=64600


/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
# ── 2. Clone AASIST repo (architecture code only — weights come from ./aasist.pth) ─
os.makedirs("./baselines", exist_ok=True)
AASIST_REPO = "./baselines/aasist"
if not os.path.exists(AASIST_REPO):
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/clovaai/aasist", AASIST_REPO],
        capture_output=True, text=True
    )
    print("✅ Cloned" if r.returncode == 0 else f"❌ {r.stderr}")
else:
    print("Already present:", AASIST_REPO)

if os.path.abspath(AASIST_REPO) not in sys.path:
    sys.path.insert(0, os.path.abspath(AASIST_REPO))
print("sys.path updated.")


Already present: ./baselines/aasist
sys.path updated.


In [4]:
# ── 3. ITW label map from meta.csv (remotezip — no full ZIP download) ─────────
def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path) as f: return f.read().strip()
    return None

HF_TOKEN  = load_hf_token()
ITW_CACHE = "./data/in_the_wild"
ZIP_URL   = "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip"
req_kwargs = {"headers": {"Authorization": f"Bearer {HF_TOKEN}"}} if HF_TOKEN else {}

print("Fetching meta.csv …")
with RemoteZip(ZIP_URL, **req_kwargs) as z:
    meta_name = next(n for n in z.namelist() if n.endswith("meta.csv"))
    with z.open(meta_name) as f:
        meta_df = pd.read_csv(io.TextIOWrapper(f))

FILE_COL  = next(c for c in meta_df.columns if "file" in c.lower() or "name" in c.lower())
LABEL_COL = next(c for c in meta_df.columns if "label" in c.lower() or "type" in c.lower())
print(f"file_col={FILE_COL!r}  label_col={LABEL_COL!r}  unique_labels={meta_df[LABEL_COL].unique().tolist()}")

REAL_VALUES = {"bona-fide", "bonafide", "real", "genuine"}
FAKE_VALUES = {"spoof", "spoofing", "fake", "deepfake"}

ITW_LABEL_MAP = {}
for _, row in meta_df.iterrows():
    fname = os.path.basename(str(row[FILE_COL]))
    lbl   = str(row[LABEL_COL]).lower().strip()
    if lbl in REAL_VALUES:   ITW_LABEL_MAP[fname] = 0
    elif lbl in FAKE_VALUES: ITW_LABEL_MAP[fname] = 1

n_real = sum(v == 0 for v in ITW_LABEL_MAP.values())
n_fake = sum(v == 1 for v in ITW_LABEL_MAP.values())
print(f"Label map: {n_real} real  +  {n_fake} fake  =  {len(ITW_LABEL_MAP)} total")

def extract_label(audio_path: str) -> int:
    fname = os.path.basename(audio_path.split("::")[0])
    if fname not in ITW_LABEL_MAP:
        raise ValueError(f"No label for: {fname!r}")
    return ITW_LABEL_MAP[fname]


Fetching meta.csv …
file_col='file'  label_col='label'  unique_labels=['spoof', 'bona-fide']
Label map: 19963 real  +  11816 fake  =  31779 total


In [5]:
# ── 4. Download ITW once + collect 5,000 balanced samples ────────────────────
# First run: ~10–15 min to download 8 GB.  Cached on every run after.

TOTAL_SAMPLES = 5_000
TARGET_REAL   = TOTAL_SAMPLES // 2
TARGET_FAKE   = TOTAL_SAMPLES - TARGET_REAL

print("Loading ITW (cached after first run) …")
ds_itw = load_dataset(
    "mueller91/In-The-Wild",
    split="train",
    cache_dir=ITW_CACHE,
    token=HF_TOKEN or None,
).cast_column("audio", Audio(sampling_rate=TARGET_SR))
print(f"Total ITW samples: {len(ds_itw)}")

real_wavs, fake_wavs     = [], []
real_labels, fake_labels = [], []

for i, row in enumerate(ds_itw):
    try:
        y = extract_label(row["audio"]["path"])
    except ValueError:
        continue

    wav = np.array(row["audio"]["array"], dtype=np.float32)

    if   y == 0 and len(real_wavs) < TARGET_REAL:
        real_wavs.append(wav);  real_labels.append(0)
    elif y == 1 and len(fake_wavs) < TARGET_FAKE:
        fake_wavs.append(wav);  fake_labels.append(1)

    if (len(real_wavs) + len(fake_wavs)) % 200 == 0:
        print(f"  [{i}]  real={len(real_wavs):4d}  fake={len(fake_wavs):4d}")

    if len(real_wavs) >= TARGET_REAL and len(fake_wavs) >= TARGET_FAKE:
        break

all_wavs   = real_wavs   + fake_wavs
all_labels = real_labels + fake_labels
print(f"\n✅ Collected: {len(real_wavs)} real  +  {len(fake_wavs)} fake  =  {len(all_wavs)} total")


Loading ITW (cached after first run) …
Total ITW samples: 31779
  [199]  real= 120  fake=  80
  [399]  real= 252  fake= 148
  [599]  real= 394  fake= 206
  [799]  real= 528  fake= 272
  [999]  real= 655  fake= 345
  [1199]  real= 793  fake= 407
  [1399]  real= 923  fake= 477
  [1599]  real=1047  fake= 553
  [1799]  real=1173  fake= 627
  [1999]  real=1299  fake= 701
  [2199]  real=1415  fake= 785
  [2399]  real=1540  fake= 860
  [2599]  real=1662  fake= 938
  [2799]  real=1789  fake=1011
  [2999]  real=1908  fake=1092
  [3199]  real=2034  fake=1166
  [3399]  real=2157  fake=1243
  [3599]  real=2281  fake=1319
  [3799]  real=2410  fake=1390
  [4075]  real=2500  fake=1500
  [4076]  real=2500  fake=1500
  [4077]  real=2500  fake=1500
  [4617]  real=2500  fake=1700
  [5151]  real=2500  fake=1900
  [5152]  real=2500  fake=1900
  [5153]  real=2500  fake=1900
  [5644]  real=2500  fake=2100
  [6167]  real=2500  fake=2300
  [6168]  real=2500  fake=2300
  [6692]  real=2500  fake=2500

✅ Collecte

In [6]:
# ── 5. InMemoryDataset + DataLoader ──────────────────────────────────────────
# FIX: num_workers=4 → num_workers=0
# InMemoryDataset stores raw numpy arrays that can't be pickled across worker
# processes on many systems (Windows always, some Linux configs).
# num_workers=0 keeps everything in-process and is safe everywhere.

def pad_or_crop(wav: torch.Tensor, length: int = CLIP_LEN) -> torch.Tensor:
    T = wav.shape[-1]
    if T >= length: return wav[..., :length]
    return torch.nn.functional.pad(wav, (0, length - T))

class InMemoryDataset(Dataset):
    def __init__(self, wavs, labels):
        self.wavs   = wavs
        self.labels = labels
    def __len__(self): return len(self.wavs)
    def __getitem__(self, idx):
        wav = torch.tensor(self.wavs[idx], dtype=torch.float32).unsqueeze(0)  # [1, T]
        return {
            "audio":       pad_or_crop(wav),
            "label":       torch.tensor(self.labels[idx], dtype=torch.long),
            "sample_rate": TARGET_SR,
        }

eval_dataset   = InMemoryDataset(all_wavs, all_labels)
# pin_memory is a no-op with num_workers=0 but harmless to keep
val_dataloader = DataLoader(eval_dataset, batch_size=32, shuffle=False,
                            num_workers=0, pin_memory=False)

s = eval_dataset[0]
print(f"Sample: audio={s['audio'].shape}  label={s['label'].item()}")
print(f"✅ DataLoader: {len(eval_dataset)} samples  |  {len(val_dataloader)} batches")


Sample: audio=torch.Size([1, 64600])  label=0
✅ DataLoader: 5000 samples  |  157 batches


In [7]:
# ── 6. Augmentation functions (mirrors validation.ipynb) ─────────────────────

def aug_pitch_up(wav, semitones=1):
    x = wav.squeeze(0).cpu().numpy()
    x = librosa.effects.pitch_shift(x, sr=TARGET_SR, n_steps=float(semitones))
    return torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(wav.device)

def aug_pitch_down(wav, semitones=1):
    return aug_pitch_up(wav, semitones=-abs(semitones))

def aug_phase_noise(wav, noise_level=0.003):
    x = wav.squeeze(0).cpu().numpy()
    D = librosa.stft(x)
    mag, phase = np.abs(D), np.angle(D)
    phase += noise_level * np.random.uniform(-np.pi, np.pi, phase.shape)
    x_out = librosa.istft(mag * np.exp(1j * phase), length=x.shape[0])
    return torch.tensor(x_out, dtype=torch.float32).unsqueeze(0).to(wav.device)

def aug_amp_scale(wav, scale=1.0):
    return wav * scale

def aug_volume(wav, gain_db=0.0):
    return wav * (10.0 ** (gain_db / 20.0))

def aug_additive_noise(wav, snr_db=40):
    sig_pwr   = wav.pow(2).mean()
    noise_pwr = sig_pwr / (10.0 ** (snr_db / 10.0))
    return wav + torch.randn_like(wav) * noise_pwr.clamp(min=1e-12).sqrt()

def aug_time_stretch(wav, rate=1.0):
    x      = wav.squeeze(0).cpu().numpy()
    target = x.shape[0]
    x      = librosa.effects.time_stretch(x, rate=rate)
    if len(x) > target: x = x[:target]
    elif len(x) < target: x = np.pad(x, (0, target - len(x)))
    return torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(wav.device)

def aug_lowpass(wav, cutoff_hz=7000):
    return torchaudio.functional.lowpass_biquad(wav, TARGET_SR, float(cutoff_hz))

def aug_highpass(wav, cutoff_hz=40):
    return torchaudio.functional.highpass_biquad(wav, TARGET_SR, float(cutoff_hz))

def aug_reverberation(wav, t60=0.5, room_scale=0.5):
    try:
        import pyroomacoustics as pra
        room_dim = [max(3.0, room_scale*10), max(3.0, room_scale*8), max(2.5, room_scale*4)]
        e_abs, max_order = pra.inverse_sabine(t60, room_dim)
        room = pra.ShoeBox(room_dim, fs=TARGET_SR, materials=pra.Material(e_abs), max_order=max_order)
        room.add_source([room_dim[0]/2, room_dim[1]/2, 1.5])
        room.add_microphone(np.c_[[room_dim[0]/2+0.5, room_dim[1]/2, 1.5]])
        room.compute_rir()
        rir = torch.tensor(room.rir[0][0], dtype=torch.float32).to(wav.device)
    except Exception:
        rir_len = max(int(t60 * TARGET_SR), 1)
        decay   = torch.exp(-torch.arange(rir_len, dtype=torch.float32) / (t60 * TARGET_SR * 0.3 + 1e-6))
        rir     = (torch.randn(rir_len) * decay).to(wav.device)
    rir = rir / (rir.abs().max() + 1e-9)
    return torch.nn.functional.conv1d(
        wav.unsqueeze(0), rir.view(1,1,-1), padding=rir.shape[0]-1
    ).squeeze(0)[..., :wav.shape[-1]]

def aug_codec_mulaw(wav):
    enc = torchaudio.functional.mu_law_encoding(wav.clamp(-1,1), quantization_channels=255)
    return torchaudio.functional.mu_law_decoding(enc, quantization_channels=255)

ATTACK_SWEEPS = {
    "pitch_up":       [{"semitones": v} for v in [1, 2, 3, 4, 5]],
    "pitch_down":     [{"semitones": v} for v in [1, 2, 3, 4, 5]],
    "phase_noise":    [{"noise_level": v} for v in [0.003, 0.006, 0.012, 0.020, 0.030]],
    "amp_scale":      [{"scale": v} for v in [0.80, 0.90, 1.00, 1.10, 1.20]],
    "volume":         [{"gain_db": v} for v in [-6.0, -3.0, 0.0, 3.0, 6.0]],
    "additive_noise": [{"snr_db": v} for v in [40, 32, 24, 16, 8]],
    "time_stretch":   [{"rate": v} for v in [0.75, 0.9, 1.07, 1.25, 1.5]],
    "lowpass":        [{"cutoff_hz": v} for v in [7000, 5000, 3500, 2500, 1500]],
    "highpass":       [{"cutoff_hz": v} for v in [40, 80, 120, 200, 350]],
    "reverberation":  [
        {"t60": 0.20, "room_scale": 0.25},
        {"t60": 0.35, "room_scale": 0.40},
        {"t60": 0.50, "room_scale": 0.50},
        {"t60": 0.70, "room_scale": 0.65},
        {"t60": 0.90, "room_scale": 0.80},
    ],
    "codec_mulaw": [{}],
}

def build_aug_fn(aug_name, params):
    dispatch = {
        "pitch_up":       lambda w: aug_pitch_up(w, **params),
        "pitch_down":     lambda w: aug_pitch_down(w, **params),
        "phase_noise":    lambda w: aug_phase_noise(w, **params),
        "amp_scale":      lambda w: aug_amp_scale(w, **params),
        "volume":         lambda w: aug_volume(w, **params),
        "additive_noise": lambda w: aug_additive_noise(w, **params),
        "time_stretch":   lambda w: aug_time_stretch(w, **params),
        "lowpass":        lambda w: aug_lowpass(w, **params),
        "highpass":       lambda w: aug_highpass(w, **params),
        "reverberation":  lambda w: aug_reverberation(w, **params),
        "codec_mulaw":    lambda w: aug_codec_mulaw(w),
    }
    return dispatch[aug_name]

class AugmentedDataset(Dataset):
    def __init__(self, dataset, aug_fn):
        self.dataset = dataset
        self.aug_fn  = aug_fn
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        wav    = sample["audio"]
        try:
            wav = pad_or_crop(self.aug_fn(wav))
        except Exception:
            pass
        return {**sample, "audio": wav}

print("✅ Augmentation functions defined.")


✅ Augmentation functions defined.


In [8]:
# ── 7. Metric helpers ────────────────────────────────────────────────────────

def compute_metrics(all_labels, all_scores) -> dict:
    labels = np.array(all_labels)
    scores = np.array(all_scores)

    # Guard: roc_auc_score raises "multi_class must be in (ovo, ovr)" when
    # scores are constant (all same value) OR labels have only one class.
    # Both happen with heavily domain-shifted models on ITW. Return nulls
    # rather than crashing the whole sweep.
    if len(np.unique(labels)) < 2:
        print("    [WARN] Only one label class present — metrics undefined.")
        return {k: None for k in
                ["val_acc","val_auc","val_eer","val_tpr","val_tnr","val_fpr","val_fnr"]}
    if len(np.unique(scores)) < 2:
        print("    [WARN] Scores are constant — model predicts same value for all inputs.")
        return {k: None for k in
                ["val_acc","val_auc","val_eer","val_tpr","val_tnr","val_fpr","val_fnr"]}

    auc    = float(roc_auc_score(labels, scores))
    fpr_c, tpr_c, thr = roc_curve(labels, scores, pos_label=1)
    fnr_c   = 1 - tpr_c
    eer_idx = int(np.nanargmin(np.abs(fnr_c - fpr_c)))
    eer     = float((fpr_c[eer_idx] + fnr_c[eer_idx]) / 2)
    eer_thr = float(thr[eer_idx])
    preds   = (scores >= eer_thr).astype(int)
    tp = int(((preds==1)&(labels==1)).sum())
    tn = int(((preds==0)&(labels==0)).sum())
    fp = int(((preds==1)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    return {
        "val_acc": (tp+tn) / max(tp+tn+fp+fn, 1),
        "val_auc": auc,
        "val_eer": eer,
        "val_tpr": tp / max(tp+fn, 1),
        "val_tnr": tn / max(tn+fp, 1),
        "val_fpr": fp / max(fp+tn, 1),
        "val_fnr": fn / max(fn+tp, 1),
    }

@torch.no_grad()
def evaluate_model(forward_fn, dataloader, device=DEVICE) -> dict:
    """
    forward_fn : callable([B, T]) → [B, 2] logits
                 InMemoryDataset returns [B, 1, T]; squeeze(1) before forward.
    """
    all_labels, all_scores = [], []
    for batch in dataloader:
        wav    = batch["audio"].squeeze(1).to(device)   # [B, 1, T] → [B, T]
        labels = batch["label"].cpu().numpy()
        out    = forward_fn(wav)
        logits = out[0] if isinstance(out, (tuple, list)) else out
        if logits.dim() == 2 and logits.shape[1] == 2:
            scores = (logits[:, 1] - logits[:, 0]).cpu().numpy()
        else:
            scores = logits.squeeze().cpu().numpy()
            if scores.ndim == 0: scores = scores.reshape(1)
        all_labels.extend(labels.tolist())
        all_scores.extend(scores.tolist())
    return compute_metrics(all_labels, all_scores)

print("✅ Metric helpers defined.")


✅ Metric helpers defined.


In [9]:
# ── 8. Load AASIST / RawNet2 (aasist.pth) ───────────────────────────────────
# FIX: aasist.pth contains keys pos_S, master1, master2, first_bn.* — this is
# RawNet2, NOT AASIST. Loading it into AASISTModel with strict=False silently
# skipped every key, leaving random weights. We now fingerprint the checkpoint
# and dispatch to the correct architecture, same as whisper.pth.
#
# RawNet2 key signatures:
#   pos_S / master1 / master2  → SincConv learnable filter params
#   first_bn.*                 → first BatchNorm on raw waveform
#   gru.*                      → GRU layer
#
# The 160-dim output is RawNet2's fc embedding before the final classifier —
# the correct output to use as the binary score is the sig_output layer [B,2].

AASIST_CKPT = "./aasist.pth"

def _is_rawnet2(keys):
    key_set = set(keys)
    return (
        any(k in key_set for k in ("pos_S", "master1", "master2")) or
        (any("sinc" in k for k in key_set) and any("first_bn" in k for k in key_set))
    ) and any("gru" in k.lower() for k in key_set)

def _is_aasist(keys):
    return any("HS_GAL" in k or "attention_weight" in k or "sinc_conv" in k
               and "GAT" in k for k in keys)

import math

class SincConv(nn.Module):
    @staticmethod
    def to_mel(hz): return 2595 * np.log10(1 + hz / 700)
    @staticmethod
    def to_hz(mel): return 700 * (10 ** (mel / 2595) - 1)

    def __init__(self, out_channels, kernel_size, sample_rate=16000,
                 min_low_hz=50, min_band_hz=50):
        super().__init__()
        self.out_channels = out_channels
        self.kernel_size  = kernel_size + (kernel_size % 2 == 0)
        self.sample_rate  = sample_rate
        self.min_low_hz   = min_low_hz
        self.min_band_hz  = min_band_hz

        low_hz  = 30.0
        high_hz = sample_rate / 2 - (min_low_hz + min_band_hz)
        mel = np.linspace(self.to_mel(low_hz), self.to_mel(high_hz), out_channels + 1)
        hz  = self.to_hz(mel)

        self.low_hz_  = nn.Parameter(torch.Tensor(hz[:-1]).view(-1, 1))
        self.band_hz_ = nn.Parameter(torch.Tensor(np.diff(hz)).view(-1, 1))

        n_lin = torch.linspace(0, (self.kernel_size / 2) - 1,
                               steps=int(self.kernel_size / 2))
        self.register_buffer("window_",
            0.54 - 0.46 * torch.cos(2 * math.pi * n_lin / self.kernel_size))
        n = (self.kernel_size - 1) / 2.0
        self.register_buffer("n_",
            2 * math.pi * torch.arange(-n, 0).view(1, -1) / sample_rate)

    def forward(self, waveforms):
        low  = self.min_low_hz + torch.abs(self.low_hz_)
        high = torch.clamp(low + self.min_band_hz + torch.abs(self.band_hz_),
                           self.min_low_hz, self.sample_rate / 2)
        band = (high - low)[:, 0]
        f_times_t_low  = torch.matmul(low,  self.n_)
        f_times_t_high = torch.matmul(high, self.n_)
        band_pass_left   = ((torch.sin(f_times_t_high) - torch.sin(f_times_t_low))
                            / (self.n_ / 2)) * self.window_
        band_pass_center = 2 * band.view(-1, 1)
        band_pass_right  = torch.flip(band_pass_left, dims=[1])
        band_pass = torch.cat([band_pass_left, band_pass_center, band_pass_right], dim=1)
        band_pass = band_pass / (2 * band[:, None])
        return F.conv1d(waveforms, band_pass.view(self.out_channels, 1, self.kernel_size),
                        padding=self.kernel_size // 2)

class FRM(nn.Module):
    def __init__(self, nb_dim):
        super().__init__()
        self.fc  = nn.Linear(nb_dim, nb_dim)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        y = F.adaptive_avg_pool1d(x, 1).view(x.size(0), -1)
        y = self.sig(self.fc(y)).view(x.size(0), x.size(1), -1)
        return x * y + y

class ResBlock(nn.Module):
    def __init__(self, nb_filts, first=False):
        super().__init__()
        self.first = first
        if not first:
            self.bn1 = nn.BatchNorm1d(nb_filts[0])
        self.lrelu = nn.LeakyReLU(negative_slope=0.3)
        self.conv1 = nn.Conv1d(nb_filts[0], nb_filts[1], 3, padding=1)
        self.bn2   = nn.BatchNorm1d(nb_filts[1])
        self.conv2 = nn.Conv1d(nb_filts[1], nb_filts[1], 3, padding=1)
        self.mp    = nn.MaxPool1d(3)
        self.frm   = FRM(nb_filts[1])
        if nb_filts[0] != nb_filts[1]:
            self.downsample  = True
            self.conv_down   = nn.Conv1d(nb_filts[0], nb_filts[1], 1)
            self.bn_down     = nn.BatchNorm1d(nb_filts[1])
        else:
            self.downsample = False

    def forward(self, x):
        identity = x
        if not self.first:
            x = self.bn1(x)
            x = self.lrelu(x)
        x = self.conv1(x)
        x = self.bn2(x)
        x = self.lrelu(x)
        x = self.conv2(x)
        if self.downsample:
            identity = self.bn_down(self.conv_down(identity))
        x = x + identity
        x = self.mp(self.frm(x))
        return x

class RawNet2(nn.Module):
    def __init__(self, filts=None, gru_hid=1024, nb_fc1=1024, nb_cls=2):
        super().__init__()
        if filts is None:
            filts = [70, [1, 32], [32, 32], [32, 64], [64, 64]]

        self.first_bn  = nn.BatchNorm1d(1)
        self.selu      = nn.SELU()
        self.sinc_conv = SincConv(filts[0], 1024, 16000)
        self.bn_sinc   = nn.BatchNorm1d(filts[0])

        self.block0 = ResBlock([filts[0], filts[1][1]], first=True)
        self.block1 = ResBlock(filts[1])
        self.block2 = ResBlock(filts[2])
        self.block3 = ResBlock(filts[3])
        self.block4 = ResBlock(filts[4])
        self.block5 = ResBlock(filts[4])

        self.bn_before_gru = nn.BatchNorm1d(filts[4][1])
        self.gru = nn.GRU(input_size=filts[4][1], hidden_size=gru_hid,
                          num_layers=1, batch_first=True)
        self.fc1_gru     = nn.Linear(gru_hid, nb_fc1)
        self.fc2_gru     = nn.Linear(nb_fc1,  nb_cls)
        self.sig_output  = nn.Linear(nb_fc1,  nb_cls)
        self.lrelu       = nn.LeakyReLU(negative_slope=0.3)
        self.drop        = nn.Dropout(0.5)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.first_bn(x)
        x = self.selu(x)
        x = self.sinc_conv(x)
        x = F.max_pool1d(torch.abs(x), 3)
        x = self.bn_sinc(x)
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.bn_before_gru(x)
        x = self.lrelu(x)
        x = x.permute(0, 2, 1)
        x, _ = self.gru(x)
        x = x[:, -1, :]
        x = self.drop(self.lrelu(self.fc1_gru(x)))
        return self.sig_output(x)          # [B, 2]

ckpt_a = torch.load(AASIST_CKPT, map_location=DEVICE)
if isinstance(ckpt_a, dict):
    sd   = ckpt_a.get("model", ckpt_a.get("state_dict", ckpt_a))
    keys = list(sd.keys())
    print(f"aasist.pth: {len(keys)} keys. First 5: {keys[:5]}")
else:
    sd   = None
    keys = []

if isinstance(ckpt_a, nn.Module):
    aasist_net = ckpt_a
    print("✅ aasist.pth is a full nn.Module")

elif _is_rawnet2(keys):
    print("🔍 Detected RawNet2 in aasist.pth  (pos_S/master1/master2 + GRU keys)")
    print("   (filename is misleading — this is NOT AASIST)")

    # Infer filter count from sinc layer weight shape if present
    sinc_key = next((k for k in keys if "sinc" in k and "weight" in k), None)
    if sinc_key:
        nb_filts_0 = sd[sinc_key].shape[0]
        print(f"   Inferred sinc out_channels={nb_filts_0} from {sinc_key!r}")
    else:
        nb_filts_0 = 70  # default

    # Infer residual block channel counts from conv weights
    def _get_filts(sd):
        """Read channel dims from block conv weights to handle non-default configs."""
        try:
            b0_out = sd[[k for k in sd if "block0" in k and "conv1.weight" in k][0]].shape[0]
            b2_out = sd[[k for k in sd if "block2" in k and "conv2.weight" in k][0]].shape[0]
            b4_out = sd[[k for k in sd if "block4" in k and "conv2.weight" in k][0]].shape[0]
            return [nb_filts_0, [1, b0_out], [b0_out, b0_out],
                    [b0_out, b2_out], [b2_out, b4_out], [b4_out, b4_out]]
        except Exception:
            return None

    filts = _get_filts(sd)
    if filts is None:
        filts = [nb_filts_0, [1, 32], [32, 32], [32, 64], [64, 64]]
        print(f"   Using default filts: {filts}")
    else:
        print(f"   Inferred filts: {filts}")

    gru_key = next((k for k in keys if "gru" in k.lower() and "weight_hh" in k), None)
    gru_hid = sd[gru_key].shape[1] if gru_key else 1024
    fc1_key = next((k for k in keys if "fc1_gru" in k and "weight" in k), None)
    nb_fc1  = sd[fc1_key].shape[0] if fc1_key else 1024
    print(f"   gru_hid={gru_hid}  nb_fc1={nb_fc1}")

    aasist_net = RawNet2(filts=filts, gru_hid=gru_hid, nb_fc1=nb_fc1)
    if all(k.startswith("model.") for k in keys[:5]):
        sd = {k[6:]: v for k, v in sd.items()}
    missing, unexpected = aasist_net.load_state_dict(sd, strict=False)
    print(f"   Loaded. Missing={len(missing)}  Unexpected={len(unexpected)}")
    if missing:    print(f"   Missing (first 5):    {missing[:5]}")
    if unexpected: print(f"   Unexpected (first 5): {unexpected[:5]}")

elif _is_aasist(keys):
    print("🔍 Detected genuine AASIST architecture")
    from models.AASIST import Model as AASISTModel
    aasist_cfg = {
        "architecture": "AASIST",
        "nb_samp":      CLIP_LEN,
        "first_conv":   128,
        "filts":        [70, [1, 32], [32, 32], [32, 64], [64, 64]],
        "gat_dims":     [64, 32],
        "pool_ratios":  [0.5, 0.7, 0.5, 0.5],
        "temperatures": [2.0, 2.0, 100.0, 100.0],
    }
    aasist_net = AASISTModel(aasist_cfg)
    missing, unexpected = aasist_net.load_state_dict(sd, strict=False)
    print(f"   Loaded. Missing={len(missing)}  Unexpected={len(unexpected)}")

else:
    raise RuntimeError(
        f"Cannot identify architecture in aasist.pth from keys: {keys[:10]}"
    )

aasist_net = aasist_net.to(DEVICE).eval()

# Verify output shape is [B, 2] before proceeding
with torch.no_grad():
    _test = aasist_net(torch.randn(2, CLIP_LEN).to(DEVICE))
    _out  = _test[0] if isinstance(_test, (tuple, list)) else _test
    assert _out.shape == (2, 2), f"Expected [2,2] output, got {_out.shape}"
print(f"✅ aasist.pth loaded and verified — output shape [B,2] confirmed")

def aasist_forward(x):
    out = aasist_net(x)
    logits = out[0] if isinstance(out, (tuple, list)) else out
    # Guard: if the model accidentally returns embeddings not logits
    if logits.shape[-1] != 2:
        raise RuntimeError(
            f"aasist_forward: expected [B,2] logits, got {logits.shape}. "
            "Check model architecture — output layer may be missing."
        )
    return logits


aasist.pth: 229 keys. First 5: ['pos_S', 'master1', 'master2', 'first_bn.weight', 'first_bn.bias']


RuntimeError: Cannot identify architecture in aasist.pth from keys: ['pos_S', 'master1', 'master2', 'first_bn.weight', 'first_bn.bias', 'first_bn.running_mean', 'first_bn.running_var', 'first_bn.num_batches_tracked', 'encoder.0.0.conv1.weight', 'encoder.0.0.conv1.bias']

In [ ]:
# ── 9. Load XLS-R (xlsr2.pt) ─────────────────────────────────────────────────
# FIX: xlsr2.pt is a valid 3 GB zip but was written with streaming/data-descriptor
# mode (general purpose bit flag 0x0008 set). PyTorch's internal PyTorchFileReader
# only handles the standard central-directory layout, so it raises:
#   "failed finding central directory"
# Python's zipfile module handles both layouts correctly, so we extract to a
# temp dir and let torch.load pick up the unpacked archive.

XLSR_ENABLED = True
XLSR_CKPT    = "./xlsr2.pt"

if XLSR_ENABLED and os.path.exists(XLSR_CKPT):
    from transformers import Wav2Vec2Model, Wav2Vec2Config
    import zipfile, tempfile, shutil

    def _load_pt_from_streaming_zip(path, map_location="cpu"):
        """
        Workaround for PyTorchFileReader rejecting zips written with
        data-descriptor mode (general purpose bit flag 0x0008).
        Extracts all members to a temp directory and calls torch.load
        on the unpacked archive root.
        """
        tmp_dir = tempfile.mkdtemp(prefix="xlsr2_unpack_")
        try:
            print(f"  Unpacking → {tmp_dir}  (may take ~30s for 3 GB) …")
            with zipfile.ZipFile(path, "r") as zf:
                zf.extractall(tmp_dir)

            members = os.listdir(tmp_dir)
            print(f"  Top-level members: {members}")

            # Standard PyTorch zip layout: single dir named after the file stem
            # containing data.pkl + data/ subdirectory
            archive_root = os.path.join(tmp_dir, members[0]) if len(members) == 1 else tmp_dir
            pkl_path = os.path.join(archive_root, "data.pkl")

            if not os.path.exists(pkl_path):
                # Some saves put data.pkl flat in the root
                pkl_candidates = [os.path.join(tmp_dir, m) for m in members if m.endswith(".pkl")]
                pkl_path = pkl_candidates[0] if pkl_candidates else None

            if pkl_path and os.path.exists(pkl_path):
                result = torch.load(pkl_path, map_location=map_location)
            else:
                result = torch.load(archive_root, map_location=map_location)

            return result
        finally:
            shutil.rmtree(tmp_dir, ignore_errors=True)

    print(f"Loading {XLSR_CKPT} ({os.path.getsize(XLSR_CKPT)/1e9:.2f} GB) …")

    try:
        ckpt_xlsr = torch.load(XLSR_CKPT, map_location="cpu")
        print("  torch.load succeeded directly.")
    except RuntimeError as e:
        if "central directory" in str(e) or "zip" in str(e).lower():
            print(f"  torch.load failed: {e}")
            print("  Falling back to zipfile extraction …")
            ckpt_xlsr = _load_pt_from_streaming_zip(XLSR_CKPT, map_location="cpu")
        else:
            raise

    # ── Identify and load the model ──────────────────────────────────────────
    if isinstance(ckpt_xlsr, nn.Module):
        xlsr_net = ckpt_xlsr
        print("✅ xlsr2.pt is a full nn.Module")
    else:
        state_dict = ckpt_xlsr.get("model", ckpt_xlsr.get("state_dict", ckpt_xlsr))
        keys       = list(state_dict.keys())
        print(f"Dict with {len(keys)} keys.  First 5: {keys[:5]}")

        has_head = any("classifier" in k or "fc" in k or "head" in k for k in keys)
        print(f"Has classification head: {has_head}")

        if has_head:
            head_key   = next(k for k in keys if "classifier" in k or "fc" in k or "head" in k)
            hidden_dim = state_dict[head_key].shape[-1]
            print(f"Detected hidden_dim={hidden_dim} from key {head_key!r}")

            class XLSRClassifier(nn.Module):
                def __init__(self, hidden_size):
                    super().__init__()
                    cfg             = Wav2Vec2Config.from_pretrained("facebook/wav2vec2-xls-r-300m")
                    self.wav2vec2   = Wav2Vec2Model(cfg)
                    self.projector  = nn.Linear(hidden_size, hidden_size)
                    self.classifier = nn.Linear(hidden_size, 2)
                def forward(self, x):
                    out    = self.wav2vec2(x).last_hidden_state
                    pooled = out.mean(dim=1)
                    return self.classifier(self.projector(pooled))

            xlsr_net = XLSRClassifier(hidden_dim)
        else:
            print("Pure backbone — adding fresh binary head.")

            class XLSRWithHead(nn.Module):
                def __init__(self):
                    super().__init__()
                    cfg           = Wav2Vec2Config.from_pretrained("facebook/wav2vec2-xls-r-300m")
                    self.backbone = Wav2Vec2Model(cfg)
                    self.head     = nn.Linear(cfg.hidden_size, 2)
                def forward(self, x):
                    out    = self.backbone(x).last_hidden_state
                    pooled = out.mean(dim=1)
                    return self.head(pooled)

            xlsr_net = XLSRWithHead()

        missing, unexpected = xlsr_net.load_state_dict(state_dict, strict=False)
        print(f"Loaded. Missing: {len(missing)}  Unexpected: {len(unexpected)}")
        if missing:    print(f"  Missing (first 5):    {missing[:5]}")
        if unexpected: print(f"  Unexpected (first 5): {unexpected[:5]}")

    xlsr_net = xlsr_net.to(DEVICE).eval()
    print(f"✅ XLS-R ready on {DEVICE}")

    def xlsr_forward(x):
        return xlsr_net(x)

elif XLSR_ENABLED and not os.path.exists(XLSR_CKPT):
    print(f"❌ {XLSR_CKPT} not found — XLS-R will be skipped.")
    def xlsr_forward(x):
        raise RuntimeError(f"xlsr2.pt not found at {XLSR_CKPT}")
else:
    def xlsr_forward(x):
        raise RuntimeError("XLSR disabled — set XLSR_ENABLED=True to activate.")
    print("ℹ️  XLS-R disabled (XLSR_ENABLED=False). Stub registered.")


In [9]:
# ── 10. Load whisper.pth — architecture-detecting loader ─────────────────────
# FIX: The previous loader assumed whisper.pth is a Whisper checkpoint.
# It is NOT — the key structure (first_bn, sinc_conv, residual blocks, GRU)
# is RawNet2.  The false Whisper detection happened because RawNet2 has a
# tensor whose name contains 'positional_embedding' with shape [?, 384],
# which collided with the Whisper size-detection heuristic.
#
# This cell now fingerprints the checkpoint by its actual keys and dispatches
# to the correct architecture.  RawNet2 is the default; a genuine Whisper
# checkpoint is also handled as a fallback.

import whisper as _whisper_lib   # kept for genuine Whisper fallback

WHISPER_CKPT = "./whisper.pth"
ckpt_w = torch.load(WHISPER_CKPT, map_location="cpu")

def _is_rawnet2(keys):
    """True if state_dict looks like RawNet2 (sinc_conv + first_bn + GRU)."""
    key_set = set(keys)
    return (
        any("sinc_conv" in k or "sinc" in k for k in key_set) and
        any("first_bn" in k for k in key_set) and
        any("gru" in k.lower() for k in key_set)
    )

def _is_whisper(keys):
    """True if state_dict has genuine Whisper encoder keys."""
    return any("encoder.conv1" in k or "encoder.blocks" in k for k in keys)

# ── Extract state dict ────────────────────────────────────────────────────────
if isinstance(ckpt_w, nn.Module):
    whisper_net = ckpt_w.to(DEVICE).eval()
    print("✅ whisper.pth is a full nn.Module — loaded directly.")

    def whisper_forward(x):
        return whisper_net(x)

else:
    sd = ckpt_w.get("model", ckpt_w.get("state_dict", ckpt_w.get("model_state_dict", ckpt_w)))
    keys = list(sd.keys())
    print(f"State dict: {len(keys)} keys.  First 5: {keys[:5]}")

    # ── Case A: RawNet2 ───────────────────────────────────────────────────────
    if _is_rawnet2(keys):
        print("\n🔍 Detected RawNet2 architecture (sinc_conv + first_bn + GRU)")
        print("   (whisper.pth name is misleading — this is NOT a Whisper checkpoint)")

        # ── Inline RawNet2 definition (mirrors clovaai/aasist repo version) ──
        class SincConv(nn.Module):
            """Learnable sinc bandpass filterbank operating on raw waveforms."""
            @staticmethod
            def to_mel(hz): return 2595 * np.log10(1 + hz / 700)
            @staticmethod
            def to_hz(mel): return 700 * (10 ** (mel / 2595) - 1)

            def __init__(self, out_channels, kernel_size, sample_rate=16000,
                         in_channels=1, stride=1, padding=0, dilation=1,
                         bias=False, groups=1, min_low_hz=50, min_band_hz=50):
                super().__init__()
                if in_channels != 1:
                    raise ValueError("SincConv only supports 1 input channel")
                self.out_channels  = out_channels
                self.kernel_size   = kernel_size + (kernel_size % 2 == 0)  # force odd
                self.stride        = stride
                self.padding       = padding
                self.dilation      = dilation
                self.sample_rate   = sample_rate
                self.min_low_hz    = min_low_hz
                self.min_band_hz   = min_band_hz

                low_hz  = 30.0
                high_hz = sample_rate / 2 - (min_low_hz + min_band_hz)
                mel  = np.linspace(self.to_mel(low_hz), self.to_mel(high_hz), out_channels + 1)
                hz   = self.to_hz(mel)

                self.low_hz_  = nn.Parameter(torch.Tensor(hz[:-1]).view(-1, 1))
                self.band_hz_ = nn.Parameter(torch.Tensor(np.diff(hz)).view(-1, 1))

                n_lin = torch.linspace(0, (self.kernel_size / 2) - 1,
                                       steps=int(self.kernel_size / 2))
                self.register_buffer("window_", 0.54 - 0.46 * torch.cos(
                    2 * math.pi * n_lin / self.kernel_size))
                n = (self.kernel_size - 1) / 2.0
                self.register_buffer("n_", 2 * math.pi * torch.arange(-n, 0).view(1, -1) / sample_rate)

            def forward(self, waveforms):
                low  = self.min_low_hz + torch.abs(self.low_hz_)
                high = torch.clamp(low + self.min_band_hz + torch.abs(self.band_hz_),
                                   self.min_low_hz, self.sample_rate / 2)
                band = (high - low)[:, 0]
                f_times_t_low  = torch.matmul(low,  self.n_)
                f_times_t_high = torch.matmul(high, self.n_)
                band_pass_left  = ((torch.sin(f_times_t_high) - torch.sin(f_times_t_low)) /
                                   (self.n_ / 2)) * self.window_
                band_pass_center = 2 * band.view(-1, 1)
                band_pass_right  = torch.flip(band_pass_left, dims=[1])
                band_pass = torch.cat([band_pass_left, band_pass_center, band_pass_right], dim=1)
                band_pass = band_pass / (2 * band[:, None])
                filters   = band_pass.view(self.out_channels, 1, self.kernel_size)
                return F.conv1d(waveforms, filters, stride=self.stride,
                                padding=self.padding, dilation=self.dilation,
                                bias=None, groups=1)

        class FRM(nn.Module):
            def __init__(self, nb_dim):
                super().__init__()
                self.fc  = nn.Linear(nb_dim, nb_dim)
                self.sig = nn.Sigmoid()
            def forward(self, x):
                y = F.adaptive_avg_pool1d(x, 1).view(x.size(0), -1)
                y = self.sig(self.fc(y)).view(x.size(0), x.size(1), -1)
                return x * y + y

        class ResBlock(nn.Module):
            def __init__(self, nb_filts, first=False):
                super().__init__()
                self.first = first
                if not first:
                    self.bn1  = nn.BatchNorm1d(nb_filts[0])
                self.lrelu = nn.LeakyReLU(negative_slope=0.3)
                self.conv1 = nn.Conv1d(nb_filts[0], nb_filts[1], 3, padding=1)
                self.bn2   = nn.BatchNorm1d(nb_filts[1])
                self.conv2 = nn.Conv1d(nb_filts[1], nb_filts[1], 3, padding=1)
                self.mp    = nn.MaxPool1d(3)
                self.frm   = FRM(nb_filts[1])
                if nb_filts[0] != nb_filts[1]:
                    self.downsample = True
                    self.conv_down  = nn.Conv1d(nb_filts[0], nb_filts[1], 1)
                    self.bn_down    = nn.BatchNorm1d(nb_filts[1])
                else:
                    self.downsample = False

            def forward(self, x):
                identity = x
                if not self.first:
                    x = self.bn1(x)
                    x = self.lrelu(x)
                x = self.conv1(x)
                x = self.bn2(x)
                x = self.lrelu(x)
                x = self.conv2(x)
                if self.downsample:
                    identity = self.bn_down(self.conv_down(identity))
                x = x + identity
                x = self.mp(self.frm(x))
                return x

        class RawNet2(nn.Module):
            def __init__(self, d_args=None):
                super().__init__()
                # Standard anti-spoofing RawNet2 config
                filts   = [70, [1, 32], [32, 32], [32, 64], [64, 64]]
                gru_hid = 1024
                nb_fc1  = 1024
                nb_cls  = 2

                self.first_bn  = nn.BatchNorm1d(1)
                self.selu      = nn.SELU()
                self.sinc_conv = SincConv(filts[0], 1024, 16000, padding=512)
                self.bn_sinc   = nn.BatchNorm1d(filts[0])

                self.block0 = ResBlock([filts[0], filts[1][1]], first=True)
                self.block1 = ResBlock(filts[1])
                self.block2 = ResBlock(filts[2])
                self.block3 = ResBlock(filts[3])
                self.block4 = ResBlock(filts[4])
                self.block5 = ResBlock(filts[4])

                self.bn_before_gru = nn.BatchNorm1d(filts[4][1])
                self.gru = nn.GRU(input_size=filts[4][1], hidden_size=gru_hid,
                                  num_layers=1, batch_first=True)
                self.fc1_gru    = nn.Linear(gru_hid, nb_fc1)
                self.fc2_gru    = nn.Linear(nb_fc1,  nb_cls)
                self.sig_output = nn.Linear(nb_fc1, nb_cls)
                self.lrelu      = nn.LeakyReLU(negative_slope=0.3)
                self.drop       = nn.Dropout(0.5)

            def forward(self, x):
                # x: [B, T]
                x = x.unsqueeze(1)                       # [B, 1, T]
                x = self.first_bn(x)
                x = self.selu(x)
                x = self.sinc_conv(x)
                x = F.max_pool1d(torch.abs(x), 3)
                x = self.bn_sinc(x)
                x = self.block0(x)
                x = self.block1(x)
                x = self.block2(x)
                x = self.block3(x)
                x = self.block4(x)
                x = self.block5(x)
                x = self.bn_before_gru(x)
                x = self.lrelu(x)
                x = x.permute(0, 2, 1)                  # [B, T', C]
                x, _ = self.gru(x)
                x = x[:, -1, :]                          # last hidden state
                x = self.drop(self.lrelu(self.fc1_gru(x)))
                return self.sig_output(x)                # [B, 2]

        whisper_net = RawNet2()
        # Strip any outer prefix (e.g. "model.") if present
        if all(k.startswith("model.") for k in keys[:5]):
            sd = {k[6:]: v for k, v in sd.items()}
        missing, unexpected = whisper_net.load_state_dict(sd, strict=False)
        print(f"   Loaded. Missing={len(missing)}  Unexpected={len(unexpected)}")
        if missing:
            print(f"   Missing (first 5): {missing[:5]}")
        if unexpected:
            print(f"   Unexpected (first 5): {unexpected[:5]}")

    # ── Case B: genuine Whisper ───────────────────────────────────────────────
    elif _is_whisper(keys):
        print("\n🔍 Detected genuine Whisper architecture")
        dim_key = next((k for k in keys if "positional_embedding" in k
                        and "encoder" in k), None)
        if dim_key:
            embed_dim    = sd[dim_key].shape[-1]
            SIZE_MAP     = {384: "tiny", 512: "base", 768: "small", 1024: "medium", 1280: "large"}
            whisper_size = SIZE_MAP.get(embed_dim, "base")
            print(f"   embed_dim={embed_dim} → Whisper '{whisper_size}'")
        else:
            whisper_size = "base"

        class WhisperClassifier(nn.Module):
            def __init__(self, size="base"):
                super().__init__()
                w            = _whisper_lib.load_model(size)
                self.encoder = w.encoder
                enc_dim      = self.encoder.positional_embedding.shape[-1]
                self.head    = nn.Linear(enc_dim, 2)
            def forward(self, x):
                mel = torchaudio.transforms.MelSpectrogram(
                    sample_rate=TARGET_SR, n_fft=400, win_length=400,
                    hop_length=160, n_mels=80).to(x.device)(x)
                mel = (mel + 1e-6).log()
                T   = mel.shape[-1]
                mel = torch.nn.functional.pad(mel, (0, max(0, 3000 - T)))[..., :3000]
                return self.head(self.encoder(mel).mean(dim=1))

        whisper_net = WhisperClassifier(whisper_size)
        def strip_prefix(d, p): return {k[len(p):]: v for k, v in d.items() if k.startswith(p)}
        enc_sd = (strip_prefix(sd, "model.encoder.") or strip_prefix(sd, "encoder.") or sd)
        missing, unexpected = whisper_net.encoder.load_state_dict(enc_sd, strict=False)
        print(f"   Encoder loaded. Missing={len(missing)}  Unexpected={len(unexpected)}")

    else:
        raise RuntimeError(
            f"Cannot identify architecture from keys: {keys[:10]}\n"
            "Expected RawNet2 (sinc_conv+first_bn+gru) or Whisper (encoder.conv1+encoder.blocks)."
        )

    whisper_net = whisper_net.to(DEVICE).eval()
    print(f"✅ whisper.pth loaded as {'RawNet2' if _is_rawnet2(keys) else 'Whisper'} on {DEVICE}")

    def whisper_forward(x):
        return whisper_net(x)


State dict: 144 keys.  First 5: ['first_bn.weight', 'first_bn.bias', 'first_bn.running_mean', 'first_bn.running_var', 'first_bn.num_batches_tracked']

🔍 Detected genuine Whisper architecture
   embed_dim=384 → Whisper 'tiny'
   Encoder loaded. Missing=67  Unexpected=144
✅ whisper.pth loaded as Whisper on cuda


In [10]:
# ── 11. Load robust_goat (WavLM + Phoneme-GAT) ───────────────────────────────
# Tries robust_goat.ckpt first (Lightning checkpoint), then robust_goat.pth.
# The checkpoint stores the full Phoneme_GAT_lit module; we unwrap to get
# the underlying nn.Module and expose a [B,T] → [B,2] forward_fn.

from phoneme_GAT.modules import Phoneme_GAT_lit   # adjust import if your path differs

ROBUST_GOAT_CKPT = "./robust_goat.ckpt"
ROBUST_GOAT_PTH  = "./robust_goat.pth"

robust_goat_net = None

if os.path.exists(ROBUST_GOAT_CKPT):
    print(f"Loading from Lightning checkpoint: {ROBUST_GOAT_CKPT}")
    try:
        robust_goat_net = Phoneme_GAT_lit.load_from_checkpoint(
            ROBUST_GOAT_CKPT, map_location=DEVICE, strict=False
        )
        print("✅ robust_goat loaded via load_from_checkpoint")
    except Exception as e:
        print(f"  load_from_checkpoint failed ({e}), falling back to torch.load …")

if robust_goat_net is None and os.path.exists(ROBUST_GOAT_PTH):
    print(f"Loading from raw checkpoint: {ROBUST_GOAT_PTH}")
    ckpt_rg = torch.load(ROBUST_GOAT_PTH, map_location=DEVICE)

    if isinstance(ckpt_rg, Phoneme_GAT_lit):
        robust_goat_net = ckpt_rg
        print("✅ robust_goat.pth is a full Phoneme_GAT_lit module")
    elif isinstance(ckpt_rg, dict):
        # Could be a Lightning state_dict dump or plain state_dict
        state_key = None
        for k in ("state_dict", "model_state_dict", "model"):
            if k in ckpt_rg:
                state_key = k; break

        if state_key:
            sd = ckpt_rg[state_key]
            print(f"  Found state_dict under key '{state_key}' ({len(sd)} keys)")
        else:
            sd = ckpt_rg
            print(f"  Treating entire dict as state_dict ({len(sd)} keys)")

        # Strip Lightning's "model." prefix if present
        if all(k.startswith("model.") for k in list(sd.keys())[:5]):
            sd = {k[len("model."):]: v for k, v in sd.items()}
            print("  Stripped 'model.' prefix from keys")

        # Instantiate with hparams stored in checkpoint or sensible defaults
        hparams = ckpt_rg.get("hyper_parameters", {})
        robust_goat_net = Phoneme_GAT_lit(**hparams) if hparams else Phoneme_GAT_lit()
        missing, unexpected = robust_goat_net.load_state_dict(sd, strict=False)
        print(f"  Loaded. Missing: {len(missing)}  Unexpected: {len(unexpected)}")
        if missing:
            print(f"  Missing keys (first 5): {missing[:5]}")
    else:
        print(f"❌ Unrecognised checkpoint type: {type(ckpt_rg)}")

if robust_goat_net is None:
    print(f"❌ Neither {ROBUST_GOAT_CKPT} nor {ROBUST_GOAT_PTH} found — robust_goat will be skipped.")
else:
    robust_goat_net = robust_goat_net.to(DEVICE).eval()
    print(f"✅ robust_goat_net on {DEVICE}, eval mode")

def robust_goat_forward(x):
    if robust_goat_net is None:
        raise RuntimeError("robust_goat not loaded.")

    with torch.no_grad():
        try:
            out = robust_goat_net._shared_pred(x)
        except Exception:
            lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
            model = robust_goat_net.model if hasattr(robust_goat_net, "model") else robust_goat_net
            out = model(x, lengths)

        if isinstance(out, dict):
            logits = out["logit"]   # <-- FORCE correct key

        elif isinstance(out, (tuple, list)):
            logits = out[1] if len(out) > 1 else out[0]

        else:
            logits = out

        if not torch.is_tensor(logits):
            raise RuntimeError(f"Expected tensor logits, got {type(logits)}")

        if logits.dim() == 1:
            logits = logits.unsqueeze(0)

        return logits

/lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Loading from Lightning checkpoint: ./robust_goat.ckpt
Now, load vocab json files from /lambda/nfs/algovirginia/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['lm_head.weight', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.bias', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])
✅ robust_goat loaded via load_from_checkpoint
✅ robust_goat_net on cuda, eval mode


In [11]:
# ── 12. Model registry ────────────────────────────────────────────────────────
MODEL_REGISTRY = [
   # ("AASIST",  aasist_forward),
  #  ("XLSR2",   xlsr_forward),
    ("Whisper", whisper_forward),
]

# Add robust_goat only if it loaded successfully
if robust_goat_net is not None:
    MODEL_REGISTRY.append(("robust_goat", robust_goat_forward))

print("Active models:", [n for n, _ in MODEL_REGISTRY])


Active models: ['Whisper', 'robust_goat']


In [12]:
# ── 13. CSV helpers ───────────────────────────────────────────────────────────
RESULTS_CSV = "itw_eval_results.csv"
CSV_COLUMNS = ["model", "aug_name", "sweep_idx", "param_json",
               "status", "error",
               "val_acc", "val_auc", "val_eer",
               "val_tpr", "val_tnr", "val_fpr", "val_fnr"]

def append_row(row: dict):
    exists = os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if not exists: w.writeheader()
        w.writerow(row)
        f.flush()

def load_completed() -> set:
    done = set()
    if not os.path.exists(RESULTS_CSV): return done
    with open(RESULTS_CSV) as f:
        for r in csv.DictReader(f):
            done.add((r["model"], r["aug_name"], r["param_json"]))
    return done

print(f"Results → {RESULTS_CSV}")


Results → itw_eval_results.csv


In [13]:
import pandas as pd

df = pd.read_csv("itw_eval_results.csv")

df = df.iloc[:53]  # keep rows 0 to 53

df.to_csv("itw_eval_results.csv", index=False)

In [ ]:
# ── 14. Main evaluation sweep ────────────────────────────────────────────────
# Resumable: already-completed (model, aug, param) triples are skipped.

completed    = load_completed()
NULL_METRICS = {k: None for k in ["val_acc","val_auc","val_eer","val_tpr","val_tnr","val_fpr","val_fnr"]}

print(f"Resuming — {len(completed)} runs already in CSV.")

for model_name, forward_fn in MODEL_REGISTRY:
    print(f"\n{'='*70}")
    print(f"  Model: {model_name}")
    print(f"{'='*70}")

    # ── Clean (no augmentation) ──────────────────────────────────────────────
    key = (model_name, "clean", "{}")
    if key not in completed:
        print(f"  [{model_name}] clean")
        try:
            m = evaluate_model(forward_fn, val_dataloader)
            append_row({"model": model_name, "aug_name": "clean", "sweep_idx": 0,
                        "param_json": "{}", "status": "ok", "error": "", **m})
            completed.add(key)
            print(f"    → AUC={m['val_auc']:.4f}  EER={m['val_eer']:.4f}  ACC={m['val_acc']:.4f}")
        except Exception as e:
            print(f"    [ERROR] {e}")
            append_row({"model": model_name, "aug_name": "clean", "sweep_idx": 0,
                        "param_json": "{}", "status": "failed", "error": str(e)[:200],
                        **NULL_METRICS})
    else:
        print(f"  [SKIP] {model_name} | clean  (already done)")

    # ── Augmentation sweep ────────────────────────────────────────────────────
    for aug_name, param_list in ATTACK_SWEEPS.items():
        for sweep_idx, params in enumerate(param_list, start=1):
            param_json = json.dumps(params, sort_keys=True)
            key        = (model_name, aug_name, param_json)

            if key in completed:
                print(f"  [SKIP] {model_name} | {aug_name} sweep {sweep_idx}  (already done)")
                continue

            print(f"  [{model_name}] {aug_name} sweep {sweep_idx} | {params}")
            aug_fn     = build_aug_fn(aug_name, params)
            aug_loader = DataLoader(
                AugmentedDataset(eval_dataset, aug_fn),
                batch_size=32, shuffle=False, num_workers=0, pin_memory=False,
            )

            try:
                m      = evaluate_model(forward_fn, aug_loader)
                status = "ok";  error = ""
                print(f"    → AUC={m['val_auc']:.4f}  EER={m['val_eer']:.4f}  ACC={m['val_acc']:.4f}")
            except Exception as e:
                print(f"    [ERROR] {e}")
                m      = NULL_METRICS;  status = "failed";  error = str(e)[:200]

            append_row({"model": model_name, "aug_name": aug_name, "sweep_idx": sweep_idx,
                        "param_json": param_json, "status": status, "error": error, **m})
            completed.add(key)

print(f"\n✅ All sweeps complete. Results saved to: {RESULTS_CSV}")


Resuming — 53 runs already in CSV.

  Model: Whisper
  [SKIP] Whisper | clean  (already done)
  [SKIP] Whisper | pitch_up sweep 1  (already done)
  [SKIP] Whisper | pitch_up sweep 2  (already done)
  [SKIP] Whisper | pitch_up sweep 3  (already done)
  [SKIP] Whisper | pitch_up sweep 4  (already done)
  [SKIP] Whisper | pitch_up sweep 5  (already done)
  [SKIP] Whisper | pitch_down sweep 1  (already done)
  [SKIP] Whisper | pitch_down sweep 2  (already done)
  [SKIP] Whisper | pitch_down sweep 3  (already done)
  [SKIP] Whisper | pitch_down sweep 4  (already done)
  [SKIP] Whisper | pitch_down sweep 5  (already done)
  [SKIP] Whisper | phase_noise sweep 1  (already done)
  [SKIP] Whisper | phase_noise sweep 2  (already done)
  [SKIP] Whisper | phase_noise sweep 3  (already done)
  [SKIP] Whisper | phase_noise sweep 4  (already done)
  [SKIP] Whisper | phase_noise sweep 5  (already done)
  [SKIP] Whisper | amp_scale sweep 1  (already done)
  [SKIP] Whisper | amp_scale sweep 2  (already d

In [ ]:
# ── 15. Summary tables ────────────────────────────────────────────────────────
# FIX: removed duplicate `import pandas as pd` (already imported in cell 2)

df = pd.read_csv(RESULTS_CSV)
ok = df[df["status"] == "ok"].copy()

# ── A. Clean baseline ────────────────────────────────────────────────────────
print("\n" + "━"*70)
print("  Clean Baseline  (no augmentation)")
print("━"*70)
clean = ok[ok["aug_name"] == "clean"][["model","val_acc","val_auc","val_eer"]].copy()
clean.columns = ["Model","ACC","AUC","EER"]
print(clean.to_string(index=False))

# ── B. Mean AUC per model × augmentation ─────────────────────────────────────
aug_only  = ok[ok["aug_name"] != "clean"]
auc_table = aug_only.groupby(["model","aug_name"])["val_auc"].mean().unstack("aug_name").round(4)
auc_table["MEAN"] = auc_table.mean(axis=1).round(4)
print("\n" + "━"*70)
print("  Mean AUC ↑  (by model × augmentation)")
print("━"*70)
print(auc_table.to_string())

# ── C. Mean EER per model × augmentation ─────────────────────────────────────
eer_table = aug_only.groupby(["model","aug_name"])["val_eer"].mean().unstack("aug_name").round(4)
eer_table["MEAN"] = eer_table.mean(axis=1).round(4)
print("\n" + "━"*70)
print("  Mean EER ↓  (by model × augmentation)")
print("━"*70)
print(eer_table.to_string())

# ── D. Degradation vs clean ───────────────────────────────────────────────────
clean_auc    = ok[ok["aug_name"]=="clean"].set_index("model")["val_auc"].to_dict()
mean_aug_auc = aug_only.groupby("model")["val_auc"].mean()
degrad       = pd.DataFrame({"Clean AUC": pd.Series(clean_auc), "Mean Aug AUC": mean_aug_auc}).round(4)
degrad["Drop"] = (degrad["Clean AUC"] - degrad["Mean Aug AUC"]).round(4)
print("\n" + "━"*70)
print("  AUC Degradation  (lower Drop = more robust)")
print("━"*70)
print(degrad.to_string())


In [ ]:
# ── 16. (Optional) Per-sweep detail for one augmentation ─────────────────────
AUG_FOCUS = "additive_noise"   # ← change to any aug name

detail = ok[ok["aug_name"] == AUG_FOCUS][["model","param_json","val_auc","val_eer"]].copy()
detail["param"] = detail["param_json"].apply(
    lambda s: list(json.loads(s).values())[0] if json.loads(s) else "—"
)
detail = detail.drop(columns="param_json").sort_values(["model","param"])
detail.columns = ["Model","AUC","EER","Param"]
print(f"\n── Per-sweep detail : {AUG_FOCUS} ──")
print(detail[["Model","Param","AUC","EER"]].to_string(index=False))
